## Section 6 — Modelisation

### Fonctions communes a tous les modeles

**Focal Loss** : penalise davantage les exemples difficiles
```
FL(pt) = alpha * (1-pt)^gamma * CE(pt)
- gamma=2 : focus sur les cas difficiles
- Sans ponderation de classe (le GAN a deja equilibre)
```

**Early Stopping sur le Recall PNEUMONIA** : on optimise
directement la metrique medicale prioritaire.

**Learning Rate Scheduler** : `ReduceLROnPlateau` sur le Recall.

### Les 5 architectures

| Modele | Parametres | Specificite |
|--------|-----------|-------------|
| M1 CNN Baseline | 25.7M | Reference — from scratch |
| M2 ResNet18 | 11.2M | Transfer Learning classique |
| M3 DenseNet-121 | 7.9M | Connexions denses — excellent medical |
| M4 EfficientNet-V2 | 21M | Meilleur ratio perf/vitesse |
| M5 CNN + ViT | variable | Hybride local+global attention |

In [ ]:
# ============================================================
# SECTION 6 : MODELISATION
# ============================================================

# ── DataLoaders avec GAN ──────────────────────────────────────
# Maintenant que les images GAN sont generees, on charge
# le dataset equilibre. Plus de WeightedRandomSampler.

train_loader, val_loader, test_loader = get_dataloaders(
    DATA_DIR, batch_size=32, avec_gan=True)

# ────────────────────────────────────────────────────────────
# 6.1 Focal Loss
# ────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    """
    Focal Loss — Lin et al. 2017 (Facebook AI Research)

    Avantage sur CrossEntropyLoss : penalise davantage les exemples
    difficiles (mal classifies). Le facteur (1-pt)^gamma reduit la
    contribution des exemples faciles.

    gamma=0  => equivalent a CrossEntropyLoss
    gamma=2  => focus fort sur les cas difficiles
    alpha=1  => pas de ponderation globale
               (le GAN a deja equilibre les classes)
    """
    def __init__(self, alpha=1.0, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        # CrossEntropy sans reduction : une valeur par exemple
        ce_loss = nn.CrossEntropyLoss(reduction="none")(inputs, targets)
        # pt = probabilite de la vraie classe
        pt = torch.exp(-ce_loss)
        # Facteur focal : reduit la contribution des exemples faciles
        focal = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal.mean()

# ────────────────────────────────────────────────────────────
# 6.2 Fonction d'entrainement commune
# ────────────────────────────────────────────────────────────

def entrainer_modele(model, train_loader, val_loader,
                     epochs=15, lr=0.001, nom_modele="modele"):
    """
    Entraine un modele avec :
    - Focal Loss (sans ponderation de classe — GAN a equilibre)
    - Optimiseur Adam + weight_decay (regularisation L2)
    - Learning Rate Scheduler sur le Recall PNEUMONIA
    - Early Stopping sur le Recall PNEUMONIA (patience=7)

    Retourne l'historique des metriques par epoch.
    """
    # Focal Loss sans ponderation (dataset equilibre par le GAN)
    criterion = FocalLoss(alpha=1.0, gamma=2.0)

    # Adam : adaptatif, converge rapidement
    # weight_decay = regularisation L2 : penalise les poids trop grands
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # ReduceLROnPlateau : divise lr par 2 si Recall stagne pendant 3 epochs
    # mode="max" car on veut maximiser le Recall
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=3, factor=0.5)

    historique = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
        "recall_pneumo": []
    }

    meilleur_recall = 0.0
    patience_es     = 7    # epochs sans amelioration avant arret
    compteur_es     = 0

    print(f"\n{'='*65}")
    print(f"ENTRAINEMENT : {nom_modele}")
    print(f"{'='*65}")
    print(f"Epochs max : {epochs} | lr : {lr} | Dataset : equilibre par GAN")
    print(f"Early Stopping : patience={patience_es} sur Recall PNEUMONIA")
    print(f"{'='*65}")

    for epoch in range(epochs):

        # ── PHASE ENTRAINEMENT ────────────────────────────────
        # model.train() : active Dropout et BatchNorm en mode train
        model.train()
        fixer_seed(epoch)  # seed different => augmentation variee

        tl, tc, tt = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()          # remettre les gradients a zero
            outputs = model(imgs)          # forward pass
            loss    = criterion(outputs, labels)  # calcul de la perte
            loss.backward()                # calcul des gradients
            optimizer.step()               # mise a jour des poids
            tl += loss.item()
            tc += (outputs.argmax(1) == labels).sum().item()
            tt += labels.size(0)

        # ── PHASE VALIDATION ──────────────────────────────────
        # model.eval() : desactive Dropout, fixe BatchNorm
        model.eval()
        vl, v_labels, v_preds, v_probs = 0, [], [], []

        # torch.no_grad() : pas de calcul de gradients => plus rapide
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs      = model(imgs)
                vl          += criterion(outputs, labels).item()
                probs        = torch.softmax(outputs, dim=1)[:, 1]
                preds        = outputs.argmax(dim=1)
                v_labels.extend(labels.cpu().numpy())
                v_preds.extend(preds.cpu().numpy())
                v_probs.extend(probs.cpu().numpy())

        # ── METRIQUES ─────────────────────────────────────────
        tla = tl / len(train_loader)
        vla = vl / len(val_loader)
        tac = tc / tt
        vac = accuracy_score(v_labels, v_preds)
        # Recall PNEUMONIA = TP / (TP + FN) — metrique prioritaire
        recall_pneumo = recall_score(v_labels, v_preds,
                                     pos_label=1, zero_division=0)

        historique["train_loss"].append(tla)
        historique["val_loss"].append(vla)
        historique["train_acc"].append(tac)
        historique["val_acc"].append(vac)
        historique["recall_pneumo"].append(recall_pneumo)

        # Ajuster le lr selon le Recall
        scheduler.step(recall_pneumo)

        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"Train Loss:{tla:.4f} Acc:{tac:.4f} | "
              f"Val Loss:{vla:.4f} Acc:{vac:.4f} | "
              f"Recall Pneumo:{recall_pneumo:.4f}")

        # ── EARLY STOPPING ────────────────────────────────────
        if recall_pneumo > meilleur_recall:
            meilleur_recall = recall_pneumo
            compteur_es     = 0
            torch.save(model.state_dict(), f"/content/{nom_modele}.pth")
            print(f"   Sauvegarde ! Meilleur Recall : {meilleur_recall:.4f}")
        else:
            compteur_es += 1
            if compteur_es >= patience_es:
                print(f"\nEarly Stopping a l'epoch {epoch+1}")
                print(f"Meilleur Recall PNEUMONIA : {meilleur_recall:.4f}")
                break

    return historique

# ────────────────────────────────────────────────────────────
# 6.3 Visualisation des courbes
# ────────────────────────────────────────────────────────────

def plot_historique(historique, titre):
    """
    Affiche 3 graphiques :
    1. Loss train vs val   => detecter l'overfitting
    2. Accuracy train vs val => performance generale
    3. Recall PNEUMONIA    => metrique medicale prioritaire
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    epochs    = range(1, len(historique["train_loss"])+1)

    # Loss : si train descend mais val monte => overfitting
    axes[0].plot(epochs, historique["train_loss"], label="Train",
                 color="steelblue", linewidth=2)
    axes[0].plot(epochs, historique["val_loss"],   label="Val",
                 color="tomato", linewidth=2)
    axes[0].set_title(f"{titre} — Loss", fontweight="bold")
    axes[0].set_xlabel("Epochs"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    ecart = historique["train_loss"][-1] - historique["val_loss"][-1]
    axes[0].text(0.05, 0.95, f"Ecart final: {ecart:+.4f}",
                 transform=axes[0].transAxes, fontsize=9, va="top",
                 color="green" if abs(ecart) < 0.1 else "red")

    # Accuracy
    axes[1].plot(epochs, historique["train_acc"], label="Train",
                 color="steelblue", linewidth=2)
    axes[1].plot(epochs, historique["val_acc"],   label="Val",
                 color="tomato", linewidth=2)
    axes[1].set_title(f"{titre} — Accuracy", fontweight="bold")
    axes[1].set_xlabel("Epochs"); axes[1].set_ylim(0.5, 1.05)
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    # Recall PNEUMONIA : la metrique qui compte en medical
    axes[2].plot(epochs, historique["recall_pneumo"],
                 color="seagreen", linewidth=2, marker="o", markersize=4,
                 label="Recall Pneumonie")
    axes[2].axhline(y=0.97, color="red", linestyle="--",
                    linewidth=1.5, label="Objectif 97%")
    meilleur_idx = np.argmax(historique["recall_pneumo"])
    meilleur_val = max(historique["recall_pneumo"])
    axes[2].scatter(meilleur_idx+1, meilleur_val, color="gold",
                    s=100, zorder=5, label=f"Meilleur: {meilleur_val:.4f}")
    axes[2].set_title(f"{titre} — Recall PNEUMONIE", fontweight="bold")
    axes[2].set_xlabel("Epochs"); axes[2].set_ylim(0.5, 1.05)
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.suptitle(titre, fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

print("Fonctions d'entrainement pretes !")
print("Dataset charge avec images GAN. Ratio NORMAL/PNEUMONIA quasi-equilibre.")

### 6.1 Modele 1 — CNN Baseline (From Scratch)

**Objectif :** Reference de base sans transfert d'apprentissage.

**Architecture :**
```
Image [3, 224, 224]
  → Conv2D(32) + BatchNorm + ReLU + MaxPool  [32, 112, 112]
  → Conv2D(64) + BatchNorm + ReLU + MaxPool  [64, 56, 56]
  → Conv2D(128) + BatchNorm + ReLU + MaxPool [128, 28, 28]
  → Flatten [100352]
  → Linear(256) + ReLU + Dropout(0.5)
  → Linear(2) => NORMAL ou PNEUMONIA
```

**Limitation attendue :** Avec peu de donnees et beaucoup de parametres,
on s'attend a de l'overfitting. Ce modele sert de point de comparaison.

In [ ]:
# ────────────────────────────────────────────────────────────
# MODELE 1 : CNN BASELINE (FROM SCRATCH)
# ────────────────────────────────────────────────────────────

fixer_seed(42)

class CNNBaseline(nn.Module):
    """
    CNN simple construit de zero.
    Sert de reference pour mesurer l'apport du Transfer Learning.
    """
    def __init__(self):
        super().__init__()
        # Bloc 1 : detecte les contours (224->112)
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        # Bloc 2 : detecte les textures (112->56)
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        # Bloc 3 : detecte les formes complexes (56->28)
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        # Classificateur final
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*28*28, 256), nn.ReLU(),
            nn.Dropout(0.5),  # anti-overfitting
            nn.Linear(256, 2) # 2 classes
        )
    def forward(self, x):
        return self.classifier(self.conv3(self.conv2(self.conv1(x))))

model1    = CNNBaseline().to(device)
nb_params = sum(p.numel() for p in model1.parameters() if p.requires_grad)
print(f"M1 CNN Baseline | Parametres : {nb_params:,}")

historique_m1 = entrainer_modele(
    model1, train_loader, val_loader,
    epochs=15, lr=0.001, nom_modele="model1_baseline")
plot_historique(historique_m1, "Modele 1 — CNN Baseline")

### 6.2 Modele 2 — ResNet18 (Transfer Learning, couches gelees)

**Concept Transfer Learning :** ResNet18 a ete pre-entraine sur 1.2 million
d'images (ImageNet). On gele toutes les couches et on remplace seulement
la derniere couche de classification.

**Avantage :** On entraine seulement ~1024 parametres au lieu de 11 millions.
Beaucoup moins d'overfitting avec peu de donnees.

In [ ]:
# ────────────────────────────────────────────────────────────
# MODELE 2 : RESNET18 TRANSFER LEARNING (COUCHES GELEES)
# ────────────────────────────────────────────────────────────

fixer_seed(42)

# Charger ResNet18 pre-entraine sur ImageNet
model2 = models.resnet18(weights="IMAGENET1K_V1")

# Geler toutes les couches : requires_grad=False
# => les poids ne seront pas mis a jour => preservation du savoir ImageNet
for param in model2.parameters():
    param.requires_grad = False

# Remplacer la derniere couche (fc)
# Originalement : Linear(512 -> 1000 classes ImageNet)
# Notre version  : Dropout + Linear(512 -> 2 classes)
model2.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model2.fc.in_features, 2)
)
model2 = model2.to(device)

total_p    = sum(p.numel() for p in model2.parameters())
entraine_p = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"M2 ResNet18 | Total: {total_p:,} | Entraines: {entraine_p:,} ({entraine_p/total_p*100:.1f}%)")

historique_m2 = entrainer_modele(
    model2, train_loader, val_loader,
    epochs=15, lr=0.001, nom_modele="model2_resnet18")
plot_historique(historique_m2, "Modele 2 — ResNet18 Transfer Learning")

### 6.3 Modele 3 — DenseNet-121

**Pourquoi DenseNet-121 pour l'imagerie medicale ?**

Dans un ResNet classique, chaque couche reçoit uniquement la sortie de la couche precedente.
Dans DenseNet, **chaque couche reçoit les sorties de TOUTES les couches precedentes** :

```
Couche 4 reçoit : sortie couche 1 + sortie couche 2 + sortie couche 3
```

Cela permet de **reutiliser les features a toutes les echelles**, ce qui est
particulierement utile pour les radios medicales ou les infiltrats peuvent
etre fins ou larges. CheXNet (Stanford) utilise DenseNet-121 pour detecter
14 pathologies pulmonaires avec des performances superieures aux radiologues.

In [ ]:
# ────────────────────────────────────────────────────────────
# MODELE 3 : DENSENET-121
# ────────────────────────────────────────────────────────────

fixer_seed(42)

# DenseNet-121 pre-entraine sur ImageNet
model3 = models.densenet121(weights="IMAGENET1K_V1")

# Geler toutes les couches sauf le classifier final
for param in model3.parameters():
    param.requires_grad = False

# DenseNet utilise model.classifier (pas model.fc comme ResNet)
# in_features = 1024 pour DenseNet-121
model3.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model3.classifier.in_features, 2)
)
model3 = model3.to(device)

total_p    = sum(p.numel() for p in model3.parameters())
entraine_p = sum(p.numel() for p in model3.parameters() if p.requires_grad)
print(f"M3 DenseNet-121 | Total: {total_p:,} | Entraines: {entraine_p:,} ({entraine_p/total_p*100:.1f}%)")
print("Architecture : connexions denses entre toutes les couches")
print("=> Reutilisation des features a toutes les echelles")
print("=> Excellent pour les infiltrats pulmonaires de tailles variees")

historique_m3 = entrainer_modele(
    model3, train_loader, val_loader,
    epochs=15, lr=0.001, nom_modele="model3_densenet121")
plot_historique(historique_m3, "Modele 3 — DenseNet-121")

### 6.4 Modele 4 — EfficientNet-V2

**Pourquoi EfficientNet-V2 ?**

EfficientNet scale simultanement trois dimensions du reseau :
- **Profondeur** : nombre de couches
- **Largeur** : nombre de filtres par couche
- **Resolution** : taille des images d'entree

EfficientNet-V2 Small est actuellement l'un des meilleurs modeles
sur ImageNet en termes de ratio performance/vitesse/parametres.
Il surpasse ResNet50 tout en etant plus rapide a entrainer.

In [ ]:
# ────────────────────────────────────────────────────────────
# MODELE 4 : EFFICIENTNET-V2
# ────────────────────────────────────────────────────────────

fixer_seed(42)

# EfficientNet-V2 Small pre-entraine sur ImageNet
model4 = models.efficientnet_v2_s(weights="IMAGENET1K_V1")

# Geler toutes les couches
for param in model4.parameters():
    param.requires_grad = False

# EfficientNet utilise model.classifier[1] pour la derniere couche
# Structure : Sequential(Dropout, Linear)
in_features = model4.classifier[1].in_features
model4.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(in_features, 2)
)
model4 = model4.to(device)

total_p    = sum(p.numel() for p in model4.parameters())
entraine_p = sum(p.numel() for p in model4.parameters() if p.requires_grad)
print(f"M4 EfficientNet-V2 | Total: {total_p:,} | Entraines: {entraine_p:,} ({entraine_p/total_p*100:.1f}%)")
print("Architecture : scaling optimal profondeur + largeur + resolution")
print("=> Meilleur ratio performance/parametres actuellement")

historique_m4 = entrainer_modele(
    model4, train_loader, val_loader,
    epochs=15, lr=0.001, nom_modele="model4_efficientnet_v2")
plot_historique(historique_m4, "Modele 4 — EfficientNet-V2")

### 6.5 Modele 5 — CNN + Vision Transformer hybride

**Pourquoi un modele hybride CNN + ViT ?**

- **CNN** : excellent pour detecter les features locales (infiltrats, opacites)
- **ViT (Vision Transformer)** : excellent pour les relations globales (attention sur l'ensemble de l'image)

La pneumonie peut toucher des zones eloignees du poumon simultanement.
Le Transformer avec son mecanisme d'attention permet de capturer ces
dependances longue distance que le CNN seul ne voit pas.

**Architecture hybride :**
```
Image → CNN (extraction features locales) → Patches
     → Transformer Encoder (attention globale)
     → Classification
```

In [ ]:
# ────────────────────────────────────────────────────────────
# MODELE 5 : CNN + VISION TRANSFORMER HYBRIDE
# ────────────────────────────────────────────────────────────

fixer_seed(42)

class HybrideCNNViT(nn.Module):
    """
    Modele hybride CNN + Vision Transformer.

    1. CNN backbone (ResNet18 gele) : extrait les features locales
       Les features spatiales sont decoupees en patches
    2. Transformer Encoder : applique l'attention sur les patches
       Capture les relations globales entre zones du poumon
    3. Classification : agregation + couche finale

    Avantage : combine la puissance des CNN pour les features locales
    et des Transformers pour les dependances longue distance.
    """
    def __init__(self, nb_classes=2, d_model=512, nhead=8,
                 num_layers=2, dropout=0.3):
        super().__init__()

        # ── CNN Backbone : ResNet18 sans la derniere couche ──
        # On utilise ResNet18 pre-entraine pour extraire les features
        resnet = models.resnet18(weights="IMAGENET1K_V1")
        # Geler les couches CNN
        for param in resnet.parameters():
            param.requires_grad = False
        # Retirer avgpool et fc : on veut les feature maps spatiales
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        # Sortie backbone : [B, 512, 7, 7] pour une image 224x224

        # ── Projection vers l'espace du Transformer ──────────
        # Convertit les features CNN en tokens pour le Transformer
        self.proj = nn.Linear(512, d_model)

        # ── Token de classification (CLS token) ──────────────
        # Token appris qui agregera l'information de tous les patches
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # ── Encodage positionnel ──────────────────────────────
        # Donne au Transformer l'information de position spatiale
        # 7*7 = 49 patches + 1 CLS token = 50 positions
        self.pos_embed = nn.Parameter(torch.randn(1, 50, d_model))

        # ── Transformer Encoder ───────────────────────────────
        # nhead    : nombre de tetes d'attention (multi-head attention)
        # num_layers : nombre de blocs Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)

        # ── Couche de normalisation et classification ─────────
        self.norm       = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, nb_classes)
        )

    def forward(self, x):
        B = x.size(0)

        # 1. Extraction features CNN : [B, 512, 7, 7]
        features = self.backbone(x)

        # 2. Aplatir les patches : [B, 49, 512]
        # 7x7 = 49 patches spatiaux
        features = features.flatten(2).transpose(1, 2)

        # 3. Projection vers l'espace Transformer : [B, 49, d_model]
        features = self.proj(features)

        # 4. Ajouter le CLS token : [B, 50, d_model]
        cls = self.cls_token.expand(B, -1, -1)
        features = torch.cat([cls, features], dim=1)

        # 5. Encodage positionnel
        features = features + self.pos_embed

        # 6. Transformer Encoder : attention globale sur tous les patches
        features = self.transformer(features)
        features = self.norm(features)

        # 7. Utiliser le CLS token pour la classification
        # Le CLS token a agregee l'information de tout l'image
        cls_output = features[:, 0]

        return self.classifier(cls_output)

model5    = HybrideCNNViT().to(device)
nb_params = sum(p.numel() for p in model5.parameters() if p.requires_grad)
print(f"M5 CNN + ViT hybride | Parametres entrainables : {nb_params:,}")
print("Architecture : CNN local + Transformer global")
print("=> Capture les infiltrats locaux ET les patterns globaux")

historique_m5 = entrainer_modele(
    model5, train_loader, val_loader,
    epochs=15, lr=0.0001,  # lr plus petit pour le Transformer
    nom_modele="model5_cnn_vit")
plot_historique(historique_m5, "Modele 5 — CNN + ViT Hybride")